In [0]:
%run "/Users/patelrahul2614@gmail.com/databrick_demo/Digital_Banking_LakeHouse_Capstone/includes"

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

# ============================================================
# TRANSACTION CLEANING
# dev.bronze.transactions -> dev.silver.silver_transactions
# ============================================================

# 1. Read from bronze layer
transaction_df = spark.table(f"{catalog}.bronze.transactions")

# 2. Drop meta columns
transaction_df = transaction_df.drop("file_name", "file_path", "ingestion_date")

# 3. Drop duplicate rows (full dupes, then dupes by transaction_id)
transaction_df = transaction_df.dropDuplicates().dropDuplicates(["transaction_id"])

# 4. Define critical columns and check nulls
critical_columns = ["transaction_id", "account_id", "transaction_date", "amount", "transaction_type", "transaction_status"]

null_counts = transaction_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in critical_columns
])
null_counts.display()

# 5. Drop rows with nulls in any critical column
transaction_df = transaction_df.na.drop(subset=critical_columns)

# 6. Additional validation
#    - Amount must be positive
transaction_df = transaction_df.filter(col("amount") > 0)
#    - Transaction date must be valid and not in the future
transaction_df = transaction_df.withColumn("transaction_date", to_date(col("transaction_date")))
transaction_df = transaction_df.filter(col("transaction_date") <= current_date())
#    - Validate transaction_type against allowed values
valid_types = ["Payment", "Withdrawal", "Transfer", "Deposit"]
transaction_df = transaction_df.filter(col("transaction_type").isin(valid_types))
#    - Validate transaction_status against allowed values
valid_statuses = ["Completed", "Failed", "Reversed"]
transaction_df = transaction_df.filter(col("transaction_status").isin(valid_statuses))

# 7. Read silver accounts for FK validation
silver_accounts_df = spark.table(f"{catalog}.silver.silver_accounts")
valid_account_ids = silver_accounts_df.select("account_id").distinct()

# 8. Flag each transaction: does account_id exist in silver_accounts?
transaction_df = (
    transaction_df
    .join(valid_account_ids.withColumn("_account_valid", lit(True)), on="account_id", how="left")
    .withColumn("_account_valid", coalesce(col("_account_valid"), lit(False)))
)

# 9. Dropped transactions: rows where account_id is NOT found in silver_accounts
dropped_transactions_df = (
    transaction_df
    .filter(~col("_account_valid"))
    .drop("_account_valid")
    .withColumn("drop_reason", lit("invalid_account_id"))
)

# 10. Clean transactions: rows where account_id exists in silver_accounts
clean_transaction_df = (
    transaction_df
    .filter(col("_account_valid"))
    .drop("_account_valid")
)

# 11. Save dropped transactions
dropped_transactions_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.dropped_transactions")

# 12. Save clean transactions
clean_transaction_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.silver_transactions")

print(f"Silver transactions saved:  {clean_transaction_df.count()}")
print(f"Dropped transactions saved: {dropped_transactions_df.count()}")

print("\n--- Dropped transactions sample ---")
dropped_transactions_df.display()

print("\n--- Clean transactions sample ---")
clean_transaction_df.display()